# Olist EDA

Business questions:
1. Order statuses: what are they and how many orders per status?
2. Revenue: `price` vs `price + freight_value`
3. Missing product categories and missing English translations

## Question 1: Order statuses

In [ ]:
import sys
sys.path.append("../src")  # let the notebook import our own package

import pandas as pd
from retailinsight import config

In [ ]:
orders = pd.read_csv(config.RAW_FILES["orders"])
orders.shape

In [ ]:
orders.head(3)

In [ ]:
orders["order_status"].isna().sum()

In [ ]:
status_counts = orders["order_status"].value_counts()
status_counts

In [ ]:
status_pct = (orders["order_status"].value_counts(normalize=True) * 100).round(2)
status_pct

In [ ]:
status_summary = pd.DataFrame({"orders": status_counts, "percent": status_pct})
status_summary

In [ ]:
orders["delivered_date_missing"] = orders["order_delivered_customer_date"].isna()
pd.crosstab(orders["order_status"], orders["delivered_date_missing"])

## Question 2: Revenue definition

**Completed Sales Revenue** = sum of `price` from order items belonging to `delivered` orders.

- Freight (`freight_value`) is kept separate.
- This is an analytical definition, not formal accounting revenue.

In [ ]:
items = pd.read_csv(config.RAW_FILES["order_items"])
items.shape

In [ ]:
items["order_id"].isin(orders["order_id"]).all()  # does every item belong to a known order?

In [ ]:
merged = items.merge(orders[["order_id", "order_status"]], on="order_id", how="left")
merged.shape

In [ ]:
merged["order_status"].isna().sum()

In [ ]:
delivered = merged[merged["order_status"] == "delivered"]
delivered.shape

In [ ]:
completed_sales_revenue = delivered["price"].sum()
all_status_price = merged["price"].sum()
completed_sales_revenue, all_status_price, all_status_price - completed_sales_revenue

In [ ]:
by_status = merged.groupby("order_status").agg(
    items=("order_id", "size"),
    orders=("order_id", "nunique"),
    price=("price", "sum"),
    freight=("freight_value", "sum"),
)
by_status["price_share_pct"] = (by_status["price"] / by_status["price"].sum() * 100).round(2)
by_status.sort_values("price", ascending=False)

In [ ]:
merged["item_total"] = merged["price"] + merged["freight_value"]
delivered = merged[merged["order_status"] == "delivered"]
delivered[["price", "freight_value", "item_total"]].sum()